## Using LLMs to predict who is the addressee based on verbal content (blind to non-verbal behavior)


In [13]:
BLUE = "\033[34m"
YELLOW = "\033[33m"
GREEN = "\033[32m"
RED = "\033[31m"
RESET = "\033[0m"

In [12]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from gradio_client import Client
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
import pickle, json, csv
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI 

In [142]:
llm = Client("osanseviero/mistral-super-fast")

response = llm.predict("hello how are you?", 0.7, 250, 0.9, 1.2, api_name="/chat")
print(response)

Loaded as API: https://osanseviero-mistral-super-fast.hf.space ✔
None


# Prepare Parsed Conversations

In [45]:
conversation_file_name = 'conversation3.txt'

prompt_conversation_1 = '''You are a domestic assistant robot named iCub. You can do several tasks, like preparing drinks and food, and your role is to help accomplish this task when required.'''

prompt_conversation_2 = '''You are a service robot named iCub, working as an assistant in shopping mall.You help customers who need information about shops inside the mall. You give informations based on customers needs and preferences.'''

prompt_conversation_3 =  '''You are a service robot named iCub, working as a waiter for a restaurant. You help customers who need to order their food and drinks.'''

In [46]:
# Read the conversation from a text file
with open(conversation_file_name, 'r') as file:
    conversation = file.readlines()

def parse_conversation(conversation): 
    parsed_conversation = []
  
    for line in conversation:
        vision_addressee = ""
        
        if ':' in line or "#" in line:
            speaker, rest = line.split(':', 1)
            message = rest.split('#', 1)[0].strip()
            ground_truth = rest.split('*Ground_Truth:', 1)[1].strip()
            
            if "I think you are talking to me" in line:
                vision_addressee = "Robot"
            elif "I guess you were talking to someone to my right" in line:
                vision_addressee = "Right"
            elif "I guess you were talking to someone to my left" in line:
                vision_addressee = "Left"

            parsed_conversation.append((speaker, message, vision_addressee, ground_truth))
            
    return parsed_conversation

parsed_conversation = parse_conversation(conversation)
print(parsed_conversation)

[('Marco', '"hi Giulia I\'m very happy to have dinner together but we have to wait also for Carlo, he is always late!"', 'Left', 'Giulia'), ('Giulia', '"hi Marco yes Carlo just texted me and he is stucked in the traffic jam but you know what? in the meanwhile we can ask the robot waiter if there are any gluten free option in the menu because Carlo is celiac."', 'Robot', 'Marco'), ('robot', "hello I'm here to assist you with any questions or requests you may have. Our restaurant is offered gluten free options including vegetable, pasta dishes, salads and some dessert. We also have a selection of gluten free breads available upon request. Let me know if there's anything specific you're interested in and I'll be happy to make recommendations.", '', 'Giulia'), ('Giulia', '"actually I was talking with Marco but thank you for answering"', 'Robot', 'robot'), ('robot', '"no problem. I apologize for misunderstanding earlier if you have any further questions or request feel free to let me know! 

In [82]:
import re

names = ['Marco', 'Giulia', 'robot', 'Robot', 'Carlo']
pattern = re.compile(r'\b(?:' + '|'.join(re.escape(name) for name in names) + r')\b')

pars_conversation_no_name = []

for i, item in enumerate(parsed_conversation):
    line = list(item)
    modified_line = pattern.sub("[]", line[1])
    line[1]= modified_line
    pars_conversation_no_name.append(tuple(line))

print(pars_conversation_no_name)   

[('Marco', '"hi [] I\'m very happy to have dinner together but we have to wait also for [], he is always late!"', 'Left', 'Giulia'), ('Giulia', '"hi [] yes [] just texted me and he is stucked in the traffic jam but you know what? in the meanwhile we can ask the [] waiter if there are any gluten free option in the menu because [] is celiac."', 'Robot', 'Marco'), ('robot', "hello I'm here to assist you with any questions or requests you may have. Our restaurant is offered gluten free options including vegetable, pasta dishes, salads and some dessert. We also have a selection of gluten free breads available upon request. Let me know if there's anything specific you're interested in and I'll be happy to make recommendations.", '', 'Giulia'), ('Giulia', '"actually I was talking with [] but thank you for answering"', 'Robot', 'robot'), ('robot', '"no problem. I apologize for misunderstanding earlier if you have any further questions or request feel free to let me know! Have a great meal."', 

# Experiments 


## Prepare ChatGPT3.5Turbo  

In [47]:
# Load environment variables from .env file
load_dotenv()

# Retrieve API key and endpoint from environment variables
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

print("API Key:", api_key)
print("API Endpoint:", api_endpoint)

# Set up the Azure OpenAI client
model_name = "gpt35turbo"
model = AzureChatOpenAI(
    api_version="2023-03-15-preview",
    deployment_name="contact-MultipartyConversation_gpt35turbo",  # Add the correct deployment name here
    model_version="0914",
    temperature= 0.1,
    max_tokens= 256,
)

API Key: 94ec2d76955a48488952c81f0d591e94
API Endpoint: https://iitlines-swecentral1.openai.azure.com/openai/deployments/contact-MultipartyConversation_gpt35turbo/chat/completions?api-version=2023-03-15-preview


## Prepare Ollama (for mistral and llama3)

In [ ]:
model_name = "mistral:7b-instruct" #"llama3"  #mistral:7b-instruct
model = OllamaLLM(model=model_name)

## Prepare template for prompt

In [48]:
binary_template = '''You are a robot taking part in a conversation with one or more people. Given who is speaking (the Speaker) and what the speaker says (the Message), identify who is the sentence's addressee. Choose among only two options: [Myself, the ROBOT], [Another person, NOT THE ROBOT] 
    
   Speaker is {speaker}. Message is {message}. 
   Answer in the format: Addressee: [option]'''

template = prompt_conversation_3 + binary_template
print(template)

You are a service robot named iCub, working as a waiter for a restaurant. You help customers who need to order their food and drinks.You are a robot taking part in a conversation with one or more people. Given who is speaking (the Speaker) and what the speaker says (the Message), identify who is the sentence's addressee. Choose among only two options: [Myself, the ROBOT], [Another person, NOT THE ROBOT] 
    
   Speaker is {speaker}. Message is {message}. 
   Answer in the format: Addressee: [option]


In [49]:
def predict_addressee(
        prompt_template:str,
        n_iteration:int, 
        save_output:bool = True):
    
    prompt = ChatPromptTemplate.from_template(prompt_template)
    chain = prompt | model

    predicted_addressee = []
    
    if save_output:
        file_name = "Exp_" + conversation_file_name.split(".")[0] + "_" + model_name + "_iteration#" + str(n_iteration) + "_with_names.csv"
        print(f"{GREEN} Saving predictions to {file_name}")
        with open(file_name, 'w', newline='') as csv_file:
            writer = csv.writer(csv_file)
            writer.writerow([prompt_template])
            # Write a header if needed
            writer.writerow(['Speaker', 'Message', 'Vision_Predicted_Addresses', 'LLM_Predicted_Addressee', "Ground_Truth_Addresses"])
    
            for speaker, message, vision_addressee, ground_truth in parsed_conversation:
                print(f"{BLUE}Speaker is: {speaker}")
                print(f"{YELLOW}Message is: {message}")
                print(f"{RED}Vision Addressee is: {vision_addressee}")
        
                response = chain.invoke({"speaker": speaker, "message": message})
                response = response.content
                predicted_addressee.append(response)
        
                print(f"{GREEN}{response}")

                # Write each response as a new row
                writer.writerow([speaker, message, vision_addressee, response, ground_truth])
      
      
# Load the list from the CSV file
def load_csv_file(file_name):
    with open(file_name, 'r') as csv_file:
        reader = csv.reader(csv_file)
        # # Skip the header
        # next(reader)
        loaded_predicted_addressee = [row[0] for row in reader]
        
    print(loaded_predicted_addressee)


In [51]:
for i in range(7,10):
    predict_addressee(template, i+1, save_output=True)

 Saving predictions to Exp_conversation3_gpt35turbo_iteration#8_with_names.csv
Speaker is: Marco
Message is: "hi Giulia I'm very happy to have dinner together but we have to wait also for Carlo, he is always late!"
Vision Addressee is: Left
Addressee: [Another person, NOT THE ROBOT]
Speaker is: Giulia
Message is: "hi Marco yes Carlo just texted me and he is stucked in the traffic jam but you know what? in the meanwhile we can ask the robot waiter if there are any gluten free option in the menu because Carlo is celiac."
Vision Addressee is: Robot
Addressee: [Another person, NOT THE ROBOT]
Speaker is: robot
Message is: hello I'm here to assist you with any questions or requests you may have. Our restaurant is offered gluten free options including vegetable, pasta dishes, salads and some dessert. We also have a selection of gluten free breads available upon request. Let me know if there's anything specific you're interested in and I'll be happy to make recommendations.
Vision Addressee is